# Session 15 · Growing and Reading Trees

Two moves today: **grow** a tree with the `max_depth` knob, and **read back** what it
leaned on — its **feature importance**, a numeric ranking of which inputs it used.

> ✏️ = your cell. Gaps never block the run.

## Step 1 · Setup — the same students, the same habits

Five habit features, `passed` as the label (still **no** `test_score` — it's the label
in disguise).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week","attendance_pct","sleep_hours_per_night","screen_time_hours_per_day","practice_sessions_per_week"]
X, y = df[habits], df['passed']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('train:', len(y_train), ' test:', len(y_test))

## Step 2 · Turn the `max_depth` knob

Fit trees of increasing depth and record **train** and **test** accuracy. Watch the
two columns start to drift apart.

In [ ]:
rows = []
for d in [1, 2, 3, 4]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    rows.append({'max_depth': d,
                 'train_acc': round(t.score(X_train, y_train), 3),
                 'test_acc':  round(t.score(X_test,  y_test),  3)})
sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))
print()
print('Training accuracy CLIMBS with depth; test accuracy barely moves.')
print('Hold that gap — Session 16 makes it the whole story.')

## Step 3 · Ask the tree what mattered — feature importance

Every fitted tree exposes `feature_importances_`: one number per feature, summing to 1,
= how much the tree relied on it. We read the **depth-3** tree.

In [ ]:
tree3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)

importance = (pd.Series(tree3.feature_importances_, index=habits)
              .sort_values(ascending=False))
print('Feature importance (depth-3 tree):')
for name, val in importance.items():
    print(f'   {name:28s} {val:.3f}')

fig, ax = plt.subplots(figsize=(8, 3.5))
importance[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_title('What did the tree lean on? (depth-3)')
ax.set_xlabel('importance (adds to 1.0)')
plt.tight_layout(); plt.show()

print()
print('study_hours ~0.88 dominates; screen_time ~0.00 was never even asked.')

## Step 4 · Cross-check the ranking against the picture

The number and the diagram should agree: the most important feature sits at the **top**
of the tree and recurs; an ignored feature appears **nowhere**.

In [ ]:
print(export_text(tree3, feature_names=habits))

fig, ax = plt.subplots(figsize=(13, 6))
plot_tree(tree3, feature_names=habits, class_names=['fail','pass'],
          filled=True, ax=ax, fontsize=8)
ax.set_title('Depth-3 tree — study_hours on top, screen_time absent')
plt.tight_layout(); plt.show()

## Step 5 · ✏️ Say it to a human

Interpretability is only useful if you can *communicate* it. Below, write **one
sentence** a school counsellor could actually use — what drives the model's prediction,
and what barely matters.

In [ ]:
# ✏️ replace the string with your plain-language sentence
counsellor_note = '...'
print(counsellor_note)

## Step 6 · ✏️ Read it critically

Importance also lets you *catch* a model relying on something it shouldn't. Imagine a
loan model whose top feature was a customer's **postcode** (a stand-in for neighbourhood,
and often for race or income).

**✏️ Your reflection (2–3 sentences):** name one feature you would be *uncomfortable*
letting a high-stakes model rely on, and explain why an importance ranking is a good
place to catch it.

*(Write your answer here, replacing this line.)*

## Wrap-up

- `max_depth` is a **knob**; deeper trees fit **training** better, test barely improves.
- `feature_importances_` **ranks the inputs** — here study_hours ~0.88, screen_time ~0.00.
- The ranking is a plain-language, contestable statement you can show a stakeholder —
  and a way to catch a model leaning on something it shouldn't.

**Next (S16):** push `max_depth` to the extreme and watch test accuracy *fall* — the
overfitting deep-dive.